# Extract lawyer annotations into gold labels

Reads two kinds of annotation and combines them into one table:
* **PDF highlights** (Priya): highlight annotations in Indian Kanoon PDFs
* **Word font colors** (Saanvi): colored text in `.docx` files

Run cells top to bottom. Update the `*_COLOR_TO_CATEGORY` dictionaries once the lawyers confirm their color keys.

In [66]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
!pip install pymupdf python-docx -q

In [68]:
import re
from pathlib import Path

import pandas as pd
import pymupdf
from docx import Document

BASE = "/content/drive/MyDrive/TrialWatch/data_analysis/categorization"
PRIYA_FOLDER = BASE + "/categorization_priya"
SAANVI_FOLDER = BASE + "/categorization_saanvi"

## 1. PDF highlights (Priya)

In [69]:
# Highlight colors in PDFs are RGB values from 0 to 1. Nearest match is used.
PDF_REFERENCE_COLORS = {
    "yellow": (0.98, 0.80, 0.35),
    "blue": (0.41, 0.69, 0.95),
    "green":  (0.49, 0.78, 0.41),
    "purple": (0.78, 0.52, 0.85),
}

COLOR_TO_CATEGORY = {
    "red": "1_elements_offense",
    "blue": "2_balance_of_speech_and_reputation",
    "green":  "3_procedural",
    "yellow": "4_insufficient_evidence",
    "purple": "5_abuse_of_process"
}

# Page furniture that Indian Kanoon adds to every page.
NOISE_PATTERNS = [
    r"Indian Kanoon - http\S+\s*\d*",
    r"\d+\s+of\s+\d+\s+\d*\s*CRM-M-\d+-\d+(\s*\(O&M\))?",
    r"\d+\s+of\s+\d+",
    r"^.{0,80}\bvs\b.{0,60}\bon \d{1,2} \w+, \d{4}",
]


def nearest_pdf_color(rgb):
    if not rgb:
        return "unknown"
    return min(PDF_REFERENCE_COLORS,
               key=lambda n: sum((a - b) ** 2 for a, b in zip(rgb, PDF_REFERENCE_COLORS[n])))


def clean(text):
    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE | re.MULTILINE)
    return re.sub(r"\s+", " ", text).strip()


def highlighted_text(page, annot):
    """Read only the text under each highlighted quad."""
    verts = annot.vertices or []
    parts = [page.get_textbox(pymupdf.Quad(verts[i:i + 4]).rect + (-1, -1, 1, 1))
             for i in range(0, len(verts), 4)]
    return " ".join(parts)


def extract_pdf(pdf_path, annotator):
    rows = []
    for page_num, page in enumerate(pymupdf.open(pdf_path), start=1):
        for annot in page.annots(types=[pymupdf.PDF_ANNOT_HIGHLIGHT]) or []:
            stroke = annot.colors.get("stroke")
            color = nearest_pdf_color(stroke)
            rows.append({
                "annotator": annotator,
                "source": "pdf_highlight",
                "file": Path(pdf_path).name,
                "page": page_num,
                "raw_color": str(tuple(round(c, 2) for c in stroke)) if stroke else None,
                "color": color,
                "category": COLOR_TO_CATEGORY.get(color, "unknown"),
                "comment": annot.info.get("content", ""),
                "text": clean(highlighted_text(page, annot)),
            })
    return rows


def extract_pdf_folder(folder, annotator):
    files = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() == ".pdf")
    df = pd.DataFrame([row for f in files for row in extract_pdf(f, annotator)])
    print(f"{annotator}: {len(files)} PDFs, {len(df)} highlights")
    if not df.empty:
        print(df.groupby(["raw_color", "color"]).size())
    return df

In [70]:
priya_df = extract_pdf_folder(PRIYA_FOLDER, annotator="Priya")
priya_df.head()

Priya: 5 PDFs, 26 highlights
raw_color           color 
(0.41, 0.69, 0.95)  blue       7
(0.49, 0.78, 0.41)  green      5
(0.78, 0.52, 0.85)  purple     1
(0.98, 0.8, 0.35)   yellow    13
dtype: int64


,annotator,source,file,page,raw_color,color,category,comment,text
0,Priya,pdf_highlight,Asif_Iqbal_Naik_vs_State_Of_J_K_Ors_on_10_Febr...,3,"(0.41, 0.69, 0.95)",blue,2_balance_of_speech_and_reputation,,6. According to learned counsel for petitioner...
1,Priya,pdf_highlight,Asif_Iqbal_Naik_vs_State_Of_J_K_Ors_on_10_Febr...,4,"(0.98, 0.8, 0.35)",yellow,4_insufficient_evidence,,9. Before appreciating the rival contentions o...
2,Priya,pdf_highlight,Asif_Iqbal_Naik_vs_State_Of_J_K_Ors_on_10_Febr...,8,"(0.41, 0.69, 0.95)",blue,2_balance_of_speech_and_reputation,,17. The case of the petitioner squarely falls ...
3,Priya,pdf_highlight,Mr_R_P_Goenka_Chairman_M_S_The_vs_State_Of_U_P...,6,"(0.98, 0.8, 0.35)",yellow,4_insufficient_evidence,,"15. Section 499 IPC provides as to what is ""de..."
4,Priya,pdf_highlight,Mr_R_P_Goenka_Chairman_M_S_The_vs_State_Of_U_P...,7,"(0.98, 0.8, 0.35)",yellow,4_insufficient_evidence,,or intended to be read or by signs or by visib...


## 2. Word font colors (Saanvi)

In [71]:
DOCX_HEX_TO_CATEGORY = {
    # Yellow
    "F1C232": "4_insufficient_evidence",
    "BF9000": "4_insufficient_evidence",
    "FFD966": "4_insufficient_evidence",
    "highlight_YELLOW (7)": "4_insufficient_evidence",


    # Green
    "38761D": "3_procedural",
    "6AA84F": "3_procedural",

    # Purple
    "674EA7": "5_abuse_of_process",
    "9900FF": "5_abuse_of_process",

    # Red
    "A61C00": "1_elements_offense",
    "FF0000": "1_elements_offense",
    "CC0000": "1_elements_offense",
    "DD7E6B": "1_elements_offense",
    "EA9999": "1_elements_offense",

    # Blue
    "1155CC": "2_balance_of_speech_and_reputation",
    "6D9EEB": "2_balance_of_speech_and_reputation"

}


def hex_to_rgb(hex_code):
    return tuple(int(hex_code[i:i + 2], 16) for i in (0, 2, 4))


def run_mark(run):
    """Highlight color, font color code, or None for plain (black or dark gray) text."""
    if run.font.highlight_color is not None:
        return f"highlight_{run.font.highlight_color}"
    color = run.font.color
    if color is not None and color.type is not None and color.rgb is not None:
        hex_code = str(color.rgb).upper()
        if max(hex_to_rgb(hex_code)) >= 80:
            return hex_code
    return None


def list_colors(docx_path):
    """Every non-black mark in a file, with a text sample."""
    found = {}
    for para in Document(docx_path).paragraphs:
        for run in para.runs:
            mark = run_mark(run)
            if mark and mark not in found:
                found[mark] = run.text[:60]
    return found


def extract_docx(docx_path, annotator):
    rows, current_mark, buffer = [], None, []

    def flush():
        text = re.sub(r"\s+", " ", "".join(buffer)).strip()
        if current_mark and text:
            category = DOCX_HEX_TO_CATEGORY.get(current_mark, f"unconfirmed_{current_mark}")
            if category == "ignore":
                return
            rows.append({
                "annotator": annotator,
                "source": "docx_font_color",
                "file": Path(docx_path).name,
                "page": None,
                "raw_color": current_mark,
                "color": category.replace("unconfirmed_", ""),
                "category": category,
                "comment": "",
                "text": text,
            })

    for para in Document(docx_path).paragraphs:
        for run in para.runs:
            mark = run_mark(run)
            if not run.text.strip() and current_mark:   # whitespace keeps a span going
                buffer.append(run.text)
                continue
            if mark != current_mark:
                flush()
                current_mark, buffer = mark, []
            if mark:
                buffer.append(run.text)
        if current_mark:
            buffer.append(" ")
    flush()
    return rows


def extract_docx_folder(folder, annotator):
    files = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() == ".docx")
    df = pd.DataFrame([row for f in files for row in extract_docx(f, annotator)])
    print(f"{annotator}: {len(files)} .docx files, {len(df)} spans")
    if not df.empty:
        df["n_words"] = df["text"].str.split().str.len()
        print(df.groupby(["category", "raw_color"]).size())
        missing = sorted(set(df["raw_color"]) - set(DOCX_HEX_TO_CATEGORY))
        if missing:
            print("Codes not in DOCX_HEX_TO_CATEGORY:", missing)
    return df

In [72]:
for f in sorted(Path(SAANVI_FOLDER).glob("*.docx")):
    print(f.name, list_colors(f))

saanvi_notes.docx {'F1C232': 'that no case is made out to quash the complaint since the pu', '674EA7': 'with', 'highlight_YELLOW (7)': ' mala fide and/', '38761D': 'him. Of course, as we have already stated,', '6AA84F': 'The Magis-', 'A61C00': 'While exercising such', 'FFD966': ' ', 'CC0000': 's rightly contended by the learned counsel for the responden', 'DD7E6B': 'us, the mere telecasting of an imputation', 'EA9999': ' it', '1155CC': 'Therefore, the trend all over the world appears to be toward', 'FF0000': 'As rightly observed by the learned Magistrate, the petitione', 'BF9000': 'f the recitals in the complaint itself alone are enough, to ', '9900FF': 'rima facie, the tweets made by the', '6D9EEB': 'the comment exemplifies that Shri'}


In [73]:
saanvi_df = extract_docx_folder(SAANVI_FOLDER, annotator="Saanvi")
saanvi_df.head()

Saanvi: 1 .docx files, 69 spans
category                            raw_color           
1_elements_offense                  A61C00                   1
                                    CC0000                   1
                                    DD7E6B                   4
                                    EA9999                   3
                                    FF0000                  11
2_balance_of_speech_and_reputation  1155CC                   2
                                    6D9EEB                   1
3_procedural                        38761D                  11
                                    6AA84F                   3
4_insufficient_evidence             BF9000                   2
                                    F1C232                   6
                                    highlight_YELLOW (7)    14
5_abuse_of_process                  674EA7                   7
                                    9900FF                   3
dtype: int64


,annotator,source,file,page,raw_color,color,category,comment,text,n_words
0,Saanvi,docx_font_color,saanvi_notes.docx,None,F1C232,4_insufficient_evidence,4_insufficient_evidence,,that no case is made out to quash the complain...,79
1,Saanvi,docx_font_color,saanvi_notes.docx,None,674EA7,5_abuse_of_process,5_abuse_of_process,,with,1
2,Saanvi,docx_font_color,saanvi_notes.docx,None,highlight_YELLOW (7),4_insufficient_evidence,4_insufficient_evidence,,mala fide and/,3
3,Saanvi,docx_font_color,saanvi_notes.docx,None,674EA7,5_abuse_of_process,5_abuse_of_process,,or where the proceedings is,5
4,Saanvi,docx_font_color,saanvi_notes.docx,None,highlight_YELLOW (7),4_insufficient_evidence,4_insufficient_evidence,,maliciously instituted,2


## 3. Combine into one gold label table

In [74]:
gold_df = pd.concat([priya_df, saanvi_df], ignore_index=True)
gold_df["n_words"] = gold_df["text"].str.split().str.len()

print(gold_df.groupby(["annotator", "category"]).size())
gold_df.to_csv(BASE + "/gold_annotations.csv", index=False)

annotator  category                          
Priya      2_balance_of_speech_and_reputation     7
           3_procedural                           5
           4_insufficient_evidence               13
           5_abuse_of_process                     1
Saanvi     1_elements_offense                    20
           2_balance_of_speech_and_reputation     3
           3_procedural                          14
           4_insufficient_evidence               22
           5_abuse_of_process                    10
dtype: int64
